# EEG_33 — Lo "switch" di fenotipo predice la decodabilità?

**Domanda di ricerca.** I fenotipi di connettività `C0` (fronto-motor) e `C1`
(fronto-occipital) sono *ortogonali* alla balanced accuracy **in media**:
between-subject forte, within-subject nullo (il *paradosso centrale* della tesi).
Tuttavia EEG_19 §9–13 mostra che l'assegnazione del fenotipo a livello **trial**
non è perfetta (ARI ≈ 0.796, non 1.0): alcuni trial di un soggetto `C0` portano
la firma di `C1` e viceversa.

> **Ipotesi (il "ponte").** Un soggetto `C0` decodifica *meglio* nei trial in cui
> adotta *transitoriamente* la strategia `C1`? Se sì, sarebbe il **primo
> collegamento fenotipo ↔ decodabilità a livello trial** — il punto in cui il
> paradosso "between forte / within nullo" potrebbe rompersi.

Coerente con EEG_25 (`C1` = tratto, strategia linguistica vincente) e con
l'ipotesi del collega (top word-length responder ≈ `C1` proficient).

**Metodo.**
- §1 Setup canonico.
- §2 Fenotipo **per-trial** via proiezione sui centroidi `C0`/`C1` (connettività
  di ampiezza `abs_pcc`, upper-triangle 1830-dim).
- §3 Correttezza per-trial via inference DHSLP da checkpoint (con fallback).
- §4 Test del ponte: *dentro* ogni soggetto, decodabilità trial allineati vs
  trial "switched" (Wilcoxon + Cohen's d — **riportiamo l'effect size**, non solo p).
- §5 Visualizzazione.
- §6 Conclusioni attese + controllo confound (SNR/varianza).

> ⚠️ Dati raw e checkpoint stanno **sul server** (env `daniele_311`, porta 8889).
> Ogni cella che dipende da artefatti server-side degrada con un messaggio chiaro
> invece di crashare.


## §1 — Setup

Header canonico del progetto: import, logger `eeg33`, config standard
(`abs_pcc`, concr4, split subject-independent), mapping word→cluster, fenotipi
`C0/C1` da `eeg16b_cluster_labels.json`.

In [ ]:
import json, logging, re, traceback
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg33')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- CONFIG STANDARD ----
N_CHANNELS     = 61
N_SAMPLES      = 384
N_CLASSES      = 4
CLUSTER_SCHEME = 'concr4'
DATA_METRIC    = 'abs_pcc'

# Iperparam DHSLP best config (EEG_13/13b)
K_WINDOWS, N_EDGES, D_MODEL, HIDDEN, N_LAYERS, DROPOUT = 8, 16, 64, 128, 2, 0.5
T_WIN = N_SAMPLES // K_WINDOWS

# word_label (0..109) -> cluster concr4 (0..3)
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}
CLUSTER_NAMES = ['CONCR', 'AZIONE', 'STATO', 'ASTR']

# ---- FENOTIPI C0/C1 (sorgente autorevole EEG_16b) ----
subj2pheno, C0_ALL, C1_ALL = {}, [], []
PHENO_PATH = project_root / 'configs' / 'eeg16b_cluster_labels.json'
PHENO_NAMES = ['C0', 'C1']
if PHENO_PATH.exists():
    _ph = json.loads(PHENO_PATH.read_text())
    subj2pheno = {int(s): int(l) for s, l in zip(_ph['subj_ids'], _ph['labels'])}
    C0_ALL = sorted([s for s, l in subj2pheno.items() if l == 0])
    C1_ALL = sorted([s for s, l in subj2pheno.items() if l == 1])
    PHENO_NAMES = _ph.get('cluster_names', PHENO_NAMES)
    log.info(f'Fenotipi caricati: |C0|={len(C0_ALL)}  |C1|={len(C1_ALL)}  (P022 escluso)')
else:
    log.warning(f'[INFO] {PHENO_PATH} assente in locale: i fenotipi verranno caricati sul server.')

# Split subject-independent standard (identico a EEG_13/26/29)
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

print('project_root =', project_root)
print('device       =', device)

### §1.1 — Sorgente dati (file `.pt` per-trial)

Struttura `data/hypergraphs_pruned_abs_pcc/P<NNN>_S<MMM>/trial_*.pt`.
Ogni `.pt` è un dict `{'x': (61,384), 'y': word_label, 'H_pruned': ...}`
(`H_pruned` ignorata). Se la root non esiste in locale degradiamo con un messaggio.

In [ ]:
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
HG_ROOT = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'

subj_sess = defaultdict(lambda: defaultdict(list))
if HG_ROOT.exists():
    for p in sorted(HG_ROOT.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if m:
            subj_sess[int(m.group(1))][int(m.group(2))].append(p)
ALL_SUBJ = sorted(subj_sess.keys())
DATA_OK = len(ALL_SUBJ) > 0
if DATA_OK:
    n_trials = sum(len(v) for s in subj_sess for v in subj_sess[s].values())
    log.info(f'Dati trovati: {len(ALL_SUBJ)} soggetti, {n_trials} trial totali.')
else:
    log.warning(f'[INFO] {HG_ROOT} assente: esegui questo notebook sul server. Le celle a valle degradano.')


def load_trial(p):
    """Ritorna (x_np(61,384) float32, word_label int)."""
    d = torch.load(p, weights_only=False)
    x = d['x'].float().numpy()
    y = d['y']
    y = int(y.squeeze()) if isinstance(y, torch.Tensor) else int(y)
    return x, y


def instance_norm(x):
    """Normalizzazione per-trial per-canale (come EEG_13b/26/29)."""
    if isinstance(x, np.ndarray):
        return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-6)
    return (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)

## §2 — Fenotipo **per-trial**

Per ogni trial calcoliamo la **connettività di ampiezza** `abs_pcc` sul *singolo
trial*: matrice 61×61 di correlazioni di Pearson assolute tra le serie temporali
dei canali, di cui prendiamo l'**upper-triangle** (1830-dim). Questa è la stessa
feature su cui EEG_16b ha definito i fenotipi a livello soggetto.

**Assegnazione (strada A — centroidi, la più semplice e robusta).**
1. Calcoliamo per ogni soggetto la sua connettività **media** (media degli upper-
   triangle dei suoi trial).
2. Costruiamo i **centroidi** `C0`/`C1` come media delle connettività medie dei
   soggetti in `C0_ALL` / `C1_ALL` (etichette autorevoli da EEG_16b).
3. Fittiamo una **PCA** sullo spazio dei soggetti (per de-rumorare / ridurre la
   dimensionalità prima della distanza) e proiettiamo centroidi e singoli trial.
4. Ogni trial è assegnato al **centroide più vicino** (distanza euclidea in spazio
   PCA → equivalente, dopo standardizzazione, alla similarità di forma).

Per ogni trial salviamo: `subj`, `session`, `pheno_subj`, `pheno_trial`,
`is_switch = (pheno_trial != pheno_subj)`.

> *(Strada B alternativa — addestrare un classificatore C0/C1 sui vettori medi
> per soggetto e applicarlo ai singoli trial — è documentata ma non usata:
> la proiezione su centroidi in spazio PCA è più stabile con pochi soggetti e
> non introduce iperparametri di regolarizzazione da tarare.)*

In [ ]:
IU = np.triu_indices(N_CHANNELS, k=1)   # 1830 indici upper-triangle


def trial_abs_pcc_vec(x_np):
    """abs_pcc del singolo trial -> vettore upper-triangle (1830,).

    x_np: (61, 384). Usiamo instance_norm per stabilità numerica, poi corrcoef."""
    xz = instance_norm(x_np)
    C = np.corrcoef(xz)                  # (61,61)
    C = np.nan_to_num(C, nan=0.0)
    return np.abs(C)[IU].astype(np.float32)


def build_trial_phenotype_table():
    """Costruisce il DataFrame per-trial con assegnazione di fenotipo.

    Ritorna (df, info) oppure (None, msg) se mancano dati/fenotipi."""
    if not DATA_OK:
        return None, 'dati .pt assenti in locale'
    if not subj2pheno:
        return None, 'fenotipi C0/C1 assenti (eeg16b_cluster_labels.json)'

    from sklearn.decomposition import PCA

    # --- 1) raccolta per-trial + accumulo medie per soggetto ---
    rows = []                       # meta per-trial
    trial_vecs = []                 # upper-triangle per-trial (allineato a rows)
    subj_sum = defaultdict(lambda: np.zeros(len(IU[0]), dtype=np.float64))
    subj_cnt = defaultdict(int)

    subj_iter = [s for s in ALL_SUBJ if s in subj2pheno]
    for sid in tqdm(subj_iter, desc='trial abs_pcc'):
        for sess, paths in subj_sess[sid].items():
            for p in paths:
                x_np, y = load_trial(p)
                v = trial_abs_pcc_vec(x_np)
                rows.append({'subj': sid, 'session': sess, 'path': str(p),
                             'y_word': y, 'cluster': label2cluster.get(y, -1),
                             'pheno_subj': subj2pheno[sid]})
                trial_vecs.append(v)
                subj_sum[sid] += v
                subj_cnt[sid] += 1

    trial_vecs = np.asarray(trial_vecs, dtype=np.float32)       # (N_trials, 1830)
    df = pd.DataFrame(rows)

    # --- 2) connettività media per soggetto ---
    subj_mean = {s: (subj_sum[s] / max(subj_cnt[s], 1)).astype(np.float32) for s in subj_iter}
    subj_ids_sorted = sorted(subj_mean.keys())
    M = np.stack([subj_mean[s] for s in subj_ids_sorted])       # (n_subj, 1830)

    # --- 3) PCA fittata sullo spazio dei soggetti (de-rumora prima delle distanze) ---
    n_comp = int(min(20, M.shape[0] - 1, M.shape[1]))
    mu = M.mean(0, keepdims=True)
    pca = PCA(n_components=n_comp, random_state=0).fit(M - mu)

    def project(V):
        return pca.transform(V - mu)

    # centroidi C0/C1 nello spazio dei soggetti, poi proiettati
    c0_subj = [s for s in subj_ids_sorted if subj2pheno[s] == 0]
    c1_subj = [s for s in subj_ids_sorted if subj2pheno[s] == 1]
    cen0 = np.stack([subj_mean[s] for s in c0_subj]).mean(0, keepdims=True)
    cen1 = np.stack([subj_mean[s] for s in c1_subj]).mean(0, keepdims=True)
    cen0_p, cen1_p = project(cen0), project(cen1)               # (1, n_comp)

    # --- 4) proiezione dei singoli trial e assegnazione al centroide piu vicino ---
    Z = project(trial_vecs)                                     # (N_trials, n_comp)
    d0 = np.linalg.norm(Z - cen0_p, axis=1)
    d1 = np.linalg.norm(Z - cen1_p, axis=1)
    df['pheno_trial'] = (d1 < d0).astype(int)                   # 0=C0-like, 1=C1-like
    df['dist_c0'] = d0
    df['dist_c1'] = d1
    # margine relativo: >0 -> piu vicino a C1, in [-1,1]
    df['c1_margin'] = (d0 - d1) / (d0 + d1 + 1e-9)
    df['is_switch'] = (df['pheno_trial'] != df['pheno_subj']).astype(int)

    info = {'n_trials': len(df), 'n_pca': n_comp,
            'switch_rate': float(df['is_switch'].mean())}
    return df, info


df_pheno, pheno_info = build_trial_phenotype_table()
if df_pheno is None:
    log.warning(f'[INFO] §2 saltato: {pheno_info}')
else:
    log.info(f"§2 ok: {pheno_info}")
    print(df_pheno.groupby(['pheno_subj', 'pheno_trial']).size().unstack(fill_value=0))
    print('switch-rate globale:', round(pheno_info['switch_rate'], 3))

## §3 — Correttezza per-trial (inference DHSLP)

Per ogni trial vogliamo `y_dec ∈ {0,1}` (decodificato correttamente o no).
La fonte autorevole sono i **checkpoint S-Spec** `models/eeg13b/P<NNN>.pt`:
carichiamo il modello del soggetto e facciamo forward sui suoi trial.

**Anti-leakage.** Un modello S-Spec è addestrato sui trial del proprio soggetto:
fare inference *in-sample* gonfia la correttezza. Per evitarlo:
- **Path preferito (no leakage):** usare i campi `labels`/`preds` del checkpoint,
  che (per costruzione EEG_13b) sono valutati sui trial di **TEST** (ultima
  sessione LOSO) — out-of-fold per quel soggetto.
- **Fallback (se i checkpoint mancano):** addestrare un DHSLP S-Spec rapido per
  soggetto in **out-of-fold** (LOSO sulle sessioni) e raccogliere le predizioni
  *solo* sui fold di validazione, così ogni trial è predetto da un modello che
  **non** l'ha visto. Documentato sotto.

In tutti i casi allineiamo `y_dec` ai trial di §2 tramite il `path` del `.pt`.

In [ ]:
class HGNNConv(nn.Module):
    """HGNN layer (Feng et al. 2019) — batched bmm, H soft dinamica."""
    def __init__(self, in_ch, out_ch, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.xavier_uniform_(self.weight)

    def forward(self, X, H):
        d_v = H.sum(dim=2).clamp(min=1e-6); d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv = (1.0 / d_v.sqrt()).unsqueeze(-1); De = (1.0 / d_e).unsqueeze(1)
        out = Dv * (X @ self.weight)
        out = torch.bmm(H.transpose(1, 2), out)
        out = De.transpose(1, 2) * out
        out = torch.bmm(H, out)
        out = Dv * out
        if self.bias is not None:
            out = out + self.bias
        return out


class DHSLP(nn.Module):
    """Dynamic Hypergraph Spectral Learning. Input x: (B,61,384)."""
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS, n_edges=N_EDGES,
                 d_model=D_MODEL, hidden=HIDDEN, n_classes=N_CLASSES,
                 n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model
        self.E = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)
        self.node_proj = nn.Sequential(nn.Linear(T_win, d_model), nn.LayerNorm(d_model), nn.ELU())
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.clf = nn.Linear(hidden, n_classes)

    def build_dynamic_H(self, feat):
        scores = torch.matmul(feat, self.E.T) / (self.d_model ** 0.5)
        return torch.softmax(scores, dim=2)

    def encode(self, x):
        B, N, T = x.shape
        outs = []
        for k in range(self.K):
            x_k = x[:, :, k*self.T_win:(k+1)*self.T_win]
            feat = self.node_proj(x_k) + self.pos_enc.unsqueeze(0)
            H = self.build_dynamic_H(feat)
            h = feat
            for conv, bn in zip(self.convs, self.bns):
                h = conv(h, H)
                h = bn(h.transpose(1, 2)).transpose(1, 2)
                h = F.elu(h); h = self.drop(h)
            outs.append(h.mean(dim=1))
        return torch.stack(outs, 0).mean(0)

    def forward(self, x):
        return self.clf(self.encode(x))

In [ ]:
CKPT_DIR = project_root / 'models' / 'eeg13b'


def _load_state_dict(ckpt):
    """Estrae lo state_dict da formati diversi di checkpoint."""
    if isinstance(ckpt, dict):
        for k in ('state_dict', 'model_state_dict', 'model'):
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
        # forse e' gia' uno state_dict puro
        if all(isinstance(v, torch.Tensor) for v in ckpt.values()):
            return ckpt
    raise KeyError('state_dict non trovato nel checkpoint')


@torch.no_grad()
def predict_subject_from_ckpt(sid):
    """Path preferito: usa labels/preds del checkpoint (out-of-fold TEST).

    Ritorna dict path-> y_dec(0/1) se possibile, altrimenti None."""
    ckpt_path = CKPT_DIR / f'P{sid:03d}.pt'
    if not ckpt_path.exists():
        return None
    ckpt = torch.load(ckpt_path, weights_only=False)
    # caso ideale: il checkpoint contiene gia' labels/preds (+ eventuale lista path)
    if isinstance(ckpt, dict) and 'labels' in ckpt and 'preds' in ckpt:
        labels = np.asarray(ckpt['labels']); preds = np.asarray(ckpt['preds'])
        ydec = (labels == preds).astype(int)
        if 'paths' in ckpt and len(ckpt['paths']) == len(ydec):
            return {str(p): int(c) for p, c in zip(ckpt['paths'], ydec)}
        # senza path espliciti: allinea per ordine ai trial della sessione di TEST
        test_sess = max(subj_sess[sid].keys()) if subj_sess[sid] else None
        if test_sess is not None:
            tp = subj_sess[sid][test_sess]
            if len(tp) == len(ydec):
                return {str(p): int(c) for p, c in zip(tp, ydec)}
        return None
    return None  # ricade nel fallback (richiede ricostruzione modello, vedi sotto)


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s)


def fallback_oof_subject(sid, max_epochs=40, lr=1e-3, batch_size=64):
    """Fallback no-leakage: DHSLP S-Spec rapido in LOSO sulle sessioni del soggetto.

    Per ogni sessione held-out, addestra sulle altre e predice gli held-out.
    Ritorna dict path-> y_dec(0/1). Costoso: usato solo se mancano i checkpoint."""
    sessions = sorted(subj_sess[sid].keys())
    if len(sessions) < 2:
        return None
    out = {}
    for held in sessions:
        tr_paths = [p for s in sessions if s != held for p in subj_sess[sid][s]]
        te_paths = list(subj_sess[sid][held])

        def make_xy(paths):
            xs, ys = [], []
            for p in paths:
                x_np, yw = load_trial(p)
                xs.append(instance_norm(x_np)); ys.append(label2cluster.get(yw, 0))
            return (torch.tensor(np.stack(xs)).float(),
                    torch.tensor(ys).long())

        Xtr, Ytr = make_xy(tr_paths)
        Xte, _ = make_xy(te_paths)
        set_seed(0)
        model = DHSLP().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        lossf = nn.CrossEntropyLoss()
        ds = torch.utils.data.TensorDataset(Xtr, Ytr)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
        model.train()
        for _ in range(max_epochs):
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(Xte.to(device)).argmax(1).cpu().numpy()
        for p, pr in zip(te_paths, preds):
            true_c = label2cluster.get(load_trial(p)[1], 0)
            out[str(p)] = int(pr == true_c)
    return out


def build_decodability(df, use_fallback=False):
    """Aggiunge la colonna y_dec a df (per i trial dove e' disponibile)."""
    if df is None:
        return None, 'tabella fenotipi assente'
    ydec_map = {}
    n_ckpt, n_fb, n_miss = 0, 0, 0
    for sid in tqdm(sorted(df['subj'].unique()), desc='decodability'):
        m = predict_subject_from_ckpt(sid)
        if m is not None:
            ydec_map.update(m); n_ckpt += 1
        elif use_fallback and DATA_OK:
            try:
                m = fallback_oof_subject(sid)
                if m is not None:
                    ydec_map.update(m); n_fb += 1
                else:
                    n_miss += 1
            except Exception as e:
                log.warning(f'fallback fallito su P{sid:03d}: {e}'); n_miss += 1
        else:
            n_miss += 1
    df = df.copy()
    df['y_dec'] = df['path'].map(ydec_map)
    info = {'subj_ckpt': n_ckpt, 'subj_fallback': n_fb, 'subj_miss': n_miss,
            'trials_with_ydec': int(df['y_dec'].notna().sum())}
    return df, info


# NB: di default NON eseguiamo il fallback (costoso). Sul server, se i checkpoint
# mancano, rilanciare con use_fallback=True.
if df_pheno is None:
    log.warning('[INFO] §3 saltato: tabella fenotipi assente.')
elif not CKPT_DIR.exists():
    log.warning(f'[INFO] Checkpoint dir {CKPT_DIR} assente. Sul server esegui:\n'
                "    df_dec, dec_info = build_decodability(df_pheno, use_fallback=True)")
    df_dec, dec_info = df_pheno.copy(), {'note': 'no ckpt, fallback non eseguito in locale'}
    df_dec['y_dec'] = np.nan
else:
    df_dec, dec_info = build_decodability(df_pheno, use_fallback=True)
    log.info(f'§3 ok: {dec_info}')

## §4 — Test del ponte

**Dentro ogni soggetto**, confrontiamo la frazione di trial decodificati
correttamente tra i trial **`C1-like`** e i trial **`C0-like`**:

$$\Delta_s = \text{acc}(\text{trial } C1\text{-like di } s) - \text{acc}(\text{trial } C0\text{-like di } s)$$

Calcoliamo `Δ_s` separatamente *dentro i soggetti C0* e *dentro i soggetti C1*.
L'ipotesi del ponte (strategia `C1` = vincente momento-per-momento) predice
**Δ_s > 0 in entrambi i gruppi**.

**Statistica.** Wilcoxon signed-rank sui `Δ_s` tra soggetti (test pareato vs 0).
Riportiamo **Cohen's d** come effect size primario: su questo dataset i p-value
sono spesso *saturi* (lezione EEG_23 §14), quindi l'effect size è la metrica
da leggere, non il p-value da solo. Richiediamo un minimo di trial per cella
(`MIN_TRIALS_PER_CELL`) per stabilità delle frazioni per-soggetto.

In [ ]:
from scipy.stats import wilcoxon

MIN_TRIALS_PER_CELL = 8   # minimo trial per (soggetto x pheno_trial) per includere il delta


def cohens_d_paired(deltas):
    """Cohen's d per campione pareato vs 0 (media/std dei delta)."""
    deltas = np.asarray(deltas, dtype=float)
    sd = deltas.std(ddof=1)
    return float(deltas.mean() / sd) if sd > 1e-12 else 0.0


def per_subject_deltas(df, subj_group_label):
    """Per i soggetti con pheno_subj==subj_group_label, calcola Delta_s.

    Delta_s = acc(C1-like trials) - acc(C0-like trials) dentro il soggetto."""
    rows = []
    sub = df[(df['pheno_subj'] == subj_group_label) & df['y_dec'].notna()]
    for sid, g in sub.groupby('subj'):
        c1 = g[g['pheno_trial'] == 1]['y_dec']
        c0 = g[g['pheno_trial'] == 0]['y_dec']
        if len(c1) >= MIN_TRIALS_PER_CELL and len(c0) >= MIN_TRIALS_PER_CELL:
            rows.append({'subj': sid, 'acc_c1like': c1.mean(), 'acc_c0like': c0.mean(),
                         'delta': c1.mean() - c0.mean(), 'n_c1': len(c1), 'n_c0': len(c0)})
    return pd.DataFrame(rows)


def bridge_test(df):
    if df is None or df['y_dec'].notna().sum() == 0:
        print('[INFO] §4 saltato: y_dec non disponibile (checkpoint server-side).')
        return None
    results = {}
    for grp, name in [(0, 'C0 subjects'), (1, 'C1 subjects')]:
        d = per_subject_deltas(df, grp)
        if len(d) < 3:
            print(f'[INFO] {name}: solo {len(d)} soggetti validi, test non affidabile.')
            results[grp] = d
            continue
        d_vals = d['delta'].values
        try:
            stat, pval = wilcoxon(d_vals)
        except ValueError:
            stat, pval = np.nan, np.nan   # tutti zero
        dcoh = cohens_d_paired(d_vals)
        print(f'\n=== {name} (n={len(d)}) ===')
        print(f'  mean Delta (acc_C1like - acc_C0like) = {d_vals.mean():+.4f}')
        print(f'  Wilcoxon stat={stat}  p={pval:.3g}')
        print(f"  Cohen's d (pareato vs 0)            = {dcoh:+.3f}   <-- effect size primario")
        d.attrs = {'mean_delta': float(d_vals.mean()), 'p': float(pval) if pval==pval else None,
                   'cohens_d': dcoh, 'n_subj': len(d)}
        results[grp] = d
    return results


bridge_results = bridge_test(df_dec) if 'df_dec' in dir() else None

## §5 — Visualizzazione

Per ogni soggetto: decodabilità dei trial **allineati** al proprio fenotipo
(`pheno_trial == pheno_subj`) vs trial **switched** (`is_switch == 1`).
- **Pannello sinistro**: scatter pareato (linee per soggetto) acc allineato vs
  acc switched, colorato per fenotipo del soggetto.
- **Pannello destro**: boxplot dei `Δ_s = acc(C1-like) − acc(C0-like)` separati
  per gruppo di soggetti `C0` / `C1` (test del ponte).

Figura salvata in `figures/eeg33_switch_decodability.png`.

In [ ]:
def plot_switch_decodability(df, bridge_results):
    if df is None or df['y_dec'].notna().sum() == 0:
        print('[INFO] §5 saltato: dati di decodabilita assenti (esegui sul server).')
        return

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # --- pannello sinistro: allineato vs switched, per soggetto ---
    ax = axes[0]
    sub = df[df['y_dec'].notna()].copy()
    sub['aligned'] = (sub['pheno_trial'] == sub['pheno_subj']).astype(int)
    pts = []
    for sid, g in sub.groupby('subj'):
        al = g[g['aligned'] == 1]['y_dec']
        sw = g[g['aligned'] == 0]['y_dec']
        if len(al) >= MIN_TRIALS_PER_CELL and len(sw) >= MIN_TRIALS_PER_CELL:
            pts.append((sid, al.mean(), sw.mean(), subj2pheno.get(sid, 0)))
    colors = {0: '#d1495b', 1: '#2a6f97'}
    for sid, a, s, ph in pts:
        ax.plot([0, 1], [a, s], '-', color=colors[ph], alpha=0.35, lw=1)
        ax.scatter([0, 1], [a, s], color=colors[ph], s=22, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['allineato\n(pheno_trial=pheno_subj)', 'switched'])
    ax.set_ylabel('frazione trial corretti (decodabilita)')
    ax.set_title(f'Decodabilita: trial allineati vs switched\n(n={len(pts)} soggetti)')
    for ph, lab in [(0, PHENO_NAMES[0]), (1, PHENO_NAMES[1])]:
        ax.scatter([], [], color=colors[ph], label=f'soggetto {lab}')
    ax.legend(loc='best', fontsize=8); ax.grid(alpha=0.2)

    # --- pannello destro: boxplot Delta_s per gruppo ---
    ax = axes[1]
    box_data, box_labels, box_colors = [], [], []
    if bridge_results:
        for grp, name in [(0, PHENO_NAMES[0]), (1, PHENO_NAMES[1])]:
            d = bridge_results.get(grp)
            if d is not None and len(d) > 0:
                box_data.append(d['delta'].values)
                box_labels.append(f'soggetti {name}\n(n={len(d)})')
                box_colors.append(colors[grp])
    if box_data:
        bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.5)
        for patch, c in zip(bp['boxes'], box_colors):
            patch.set_facecolor(c); patch.set_alpha(0.4)
        for i, dd in enumerate(box_data):
            ax.scatter(np.full(len(dd), i+1) + np.random.uniform(-0.08, 0.08, len(dd)),
                       dd, color=box_colors[i], s=18, zorder=3, alpha=0.8)
    ax.axhline(0, color='k', lw=1, ls='--')
    ax.set_ylabel(r'$\Delta_s$ = acc(C1-like) - acc(C0-like)')
    ax.set_title('Test del ponte: switch C1 -> migliore decodabilita?\n(Delta>0 supporta l\'ipotesi)')
    ax.grid(alpha=0.2)

    plt.tight_layout()
    out = FIG_DIR / 'eeg33_switch_decodability.png'
    fig.savefig(out, dpi=140, bbox_inches='tight')
    print('Figura salvata in', out)
    plt.show()


plot_switch_decodability(df_dec if 'df_dec' in dir() else None,
                         bridge_results if 'bridge_results' in dir() else None)

### §5.1 — Controllo confound (SNR / varianza)

Rischio: i trial `C1-like` potrebbero decodificare meglio semplicemente perché
sono **meno rumorosi** (la matrice di correlazione di un trial pulito assomiglia
di più a un centroide stabile). Confrontiamo allora **varianza media** e una
proxy di **SNR** (rapporto tra ampiezza della banda di segnale e rumore ad alta
frequenza) tra trial `C0-like` e `C1-like`. Se l'SNR è ben bilanciato, l'eventuale
effetto sul ponte **non** è spiegato dalla qualità del segnale.

In [ ]:
def trial_quality_metrics(x_np):
    """Ritorna (var_mean, snr_proxy) per un trial (61,384).

    snr_proxy: potenza nelle differenze a bassa frequenza (segnale lento) vs
    potenza nelle differenze prime (rumore ad alta freq), media sui canali."""
    var_mean = float(np.mean(np.var(x_np, axis=1)))
    diff = np.diff(x_np, axis=1)
    hf_pow = np.mean(diff ** 2, axis=1) + 1e-9          # proxy rumore HF
    sig_pow = np.var(x_np, axis=1)                      # potenza totale
    snr = float(np.mean(sig_pow / hf_pow))
    return var_mean, snr


def confound_check(df, max_per_group=4000):
    if df is None or not DATA_OK:
        print('[INFO] §5.1 saltato: dati .pt assenti.')
        return
    sub = df.copy()
    rng = np.random.default_rng(0)
    recs = []
    for ph in (0, 1):
        paths = sub[sub['pheno_trial'] == ph]['path'].values
        if len(paths) > max_per_group:
            paths = rng.choice(paths, max_per_group, replace=False)
        for p in tqdm(paths, desc=f'quality pheno_trial={ph}'):
            x_np, _ = load_trial(Path(p))
            v, s = trial_quality_metrics(x_np)
            recs.append({'pheno_trial': ph, 'var': v, 'snr': s})
    q = pd.DataFrame(recs)
    if q.empty:
        print('[INFO] nessun trial campionato.')
        return
    from scipy.stats import mannwhitneyu
    for col in ('var', 'snr'):
        a = q[q['pheno_trial'] == 0][col]; b = q[q['pheno_trial'] == 1][col]
        try:
            _, pval = mannwhitneyu(a, b)
        except ValueError:
            pval = np.nan
        # Cohen's d unpaired
        ns = (len(a) + len(b) - 2)
        sp = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / max(ns, 1))
        d = (b.mean() - a.mean()) / sp if sp > 1e-12 else 0.0
        print(f'{col:>4s}: C0-like={a.mean():.4g}  C1-like={b.mean():.4g}  '
              f"MWU p={pval:.3g}  Cohen's d={d:+.3f}")
    print('\nInterpretazione: |d| piccolo (<0.2) su var/SNR => qualita bilanciata, '
          'il ponte NON e spiegato dal rumore.')


confound_check(df_pheno if 'df_pheno' in dir() else None)

## §6 — Conclusioni attese / Come leggere i risultati

**Lettura del test del ponte (§4).** Riferimento primario: **Cohen's d** sui
`Δ_s = acc(C1-like) − acc(C0-like)` (non il p-value, saturo su questo dataset).

| Esito | Interpretazione |
|-------|-----------------|
| **Δ > 0 con d ≥ ~0.3 in *entrambi* i gruppi** (C0 *e* C1) | 🟢 **Risultato forte e nuovo.** I trial `C1-like` decodificano meglio *anche dentro i soggetti C0*: la strategia linguistica `C1` è quella vincente **momento-per-momento**. È il **primo ponte** fenotipo↔decodabilità a livello trial — il punto in cui il paradosso "between forte / within nullo" si rompe. Coerente con EEG_25 e con l'ipotesi del collega (top word-length responder ≈ C1). |
| Δ ≈ 0, \|d\| piccolo in entrambi i gruppi | ⚪ **Null informativo.** Il paradosso regge **anche a livello trial**: il fenotipo resta ortogonale alla decodabilità a ogni scala. Completa la triangolazione (between forte, within nullo, *trial nullo*). |
| Δ con segno **opposto** tra i due gruppi | 🟡 Effetto specifico del soggetto, non una strategia universale: probabile *regression-to-the-mean* o artefatto di assegnazione — interpretare con cautela. |

**Controllo confound obbligatorio (§5.1).** La conclusione 🟢 è valida **solo se**
var/SNR sono bilanciati tra trial `C0-like` e `C1-like` (\|Cohen's d\| < ~0.2). Se i
trial `C1-like` sono sistematicamente meno rumorosi, l'effetto sul ponte potrebbe
essere un artefatto di **qualità del segnale**, non di strategia neurale — in tal
caso il risultato va declassato a correlazione qualità↔assegnazione.

**Caveat metodologici.**
- L'assegnazione per-trial (§2) è basata su una proiezione su centroidi: ARI≈0.796
  (EEG_19) implica rumore di assegnazione che *attenua* l'effetto (bias verso il
  null), quindi un effetto positivo è conservativo.
- I `Δ_s` richiedono ≥ `MIN_TRIALS_PER_CELL` trial per cella: soggetti molto
  sbilanciati (pochissimi switch) sono esclusi, riducendo n.
- La correttezza per-trial deve essere **out-of-fold** (§3) per evitare leakage;
  in locale non è calcolata (checkpoint/dati server-side).

**Prossimo passo se 🟢:** condizionare il ponte allo schema `gram4` (la strategia
`C1` è linguistica → l'effetto dovrebbe essere più forte sulle parole con
struttura morfo-sintattica più ricca) e correlare il `c1_margin` continuo (§2)
con la probabilità di decodifica corretta (regressione logistica a effetti misti
per-soggetto).